# 06 - pycaret v2 model

This will be the model used for the final model, since due to its accurate precision

Author: José Fernando Gutiérrez Montero

In [1]:
# Cell 1: Imports y configuración inicial
import pandas as pd
from pycaret.regression import *
import os

MODEL_PATH = './../../data/trained_models/electricity/'
os.makedirs(MODEL_PATH, exist_ok=True)

In [12]:
# Cell 2: Cargar datos
data_path = './../../data/electricity/processed/demand_model_ready.parquet'
df = pd.read_parquet(data_path)

# Seleccionamos solo las columnas que queremos usar hasta IFA_FLOW
columns_to_use = [
    'SETTLEMENT_DATE', 'SETTLEMENT_PERIOD', 'ND', 'TSD',
    'EMBEDDED_WIND_GENERATION','EMBEDDED_WIND_CAPACITY','EMBEDDED_SOLAR_GENERATION',
    'EMBEDDED_SOLAR_CAPACITY','NON_BM_STOR','PUMP_STORAGE_PUMPING','SCOTTISH_TRANSFER','IFA_FLOW'
]
df = df[columns_to_use]

# Convertimos SETTLEMENT_DATE a datetime si no lo está
# Cell 3: Preprocesamiento de features (sin tocar frontend)
# Convertimos SETTLEMENT_DATE a datetime
df['SETTLEMENT_DATE'] = pd.to_datetime(df['SETTLEMENT_DATE'])

# Generamos variables temporales automáticamente
df['Year'] = df['SETTLEMENT_DATE'].dt.year
df['Month'] = df['SETTLEMENT_DATE'].dt.month
df['Day'] = df['SETTLEMENT_DATE'].dt.day
df['Weekday'] = df['SETTLEMENT_DATE'].dt.weekday  # opcional, lunes=0

# Creamos df_model para PyCaret y quitamos la columna original de fecha
df_model = df.drop(columns=['SETTLEMENT_DATE'])

In [15]:
reg = setup(
    data=df_model,
    target='ND',
    session_id=123,
    numeric_features=['TSD','EMBEDDED_WIND_GENERATION','EMBEDDED_WIND_CAPACITY',
                    'EMBEDDED_SOLAR_GENERATION','EMBEDDED_SOLAR_CAPACITY','NON_BM_STOR','PUMP_STORAGE_PUMPING',
                    'SCOTTISH_TRANSFER','IFA_FLOW','Year','Month','Day','Weekday'],
    fold_strategy='timeseries',
    fold=5,
    transform_target=True,
    data_split_shuffle=False,   # <--- importante
    fold_shuffle=False          # <--- importante
)

,Description,Value
0,Session id,123
1,Target,ND
2,Target type,Regression
3,Original data shape,"(434590, 15)"
4,Transformed data shape,"(434590, 15)"
5,Transformed train set shape,"(304213, 15)"
6,Transformed test set shape,"(130377, 15)"
7,Numeric features,13
8,Categorical features,1
9,Preprocess,True


In [16]:
# Cell 5: Comparar modelos y elegir mejor
best_model = compare_models(sort='RMSE', n_select=3)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,773.4364,1847039.0869,1084.3028,0.9688,0.0305,0.0213,4.0160
rf,Random Forest Regressor,852.2097,2517365.7045,1234.6808,0.9576,0.0340,0.0232,7.7020
xgboost,Extreme Gradient Boosting,910.1593,2049825.3775,1247.7551,0.9652,0.0367,0.0259,0.2820
lightgbm,Light Gradient Boosting Machine,1003.6553,2386635.8947,1364.0846,0.9594,0.0401,0.0286,0.6660
dt,Decision Tree Regressor,933.8745,3149337.9767,1394.5491,0.9469,0.0384,0.0254,0.4620
knn,K Neighbors Regressor,1175.6741,7385182.5667,1596.8520,0.8757,0.0438,0.0319,0.2380
gbr,Gradient Boosting Regressor,1561.1988,4389647.9568,2048.7244,0.9232,0.0598,0.0460,5.0740
ada,AdaBoost Regressor,2979.4241,13390567.0055,3654.2338,0.7621,0.1109,0.0929,3.0440
en,Elastic Net,3742.1605,21843461.0764,4657.3263,0.6137,0.1384,0.1111,0.4040
lasso,Lasso Regression,3813.8025,22576027.1556,4732.1125,0.5996,0.1397,0.1144,0.5800


In [17]:
df_model.corr()['ND'].sort_values(ascending=False)

ND                           1.000000
SETTLEMENT_PERIOD            0.384361
TSD                          0.186807
NON_BM_STOR                  0.106365
Day                         -0.022326
Month                       -0.094376
IFA_FLOW                    -0.094560
Weekday                     -0.185316
SCOTTISH_TRANSFER           -0.200314
EMBEDDED_SOLAR_GENERATION   -0.248286
EMBEDDED_WIND_GENERATION    -0.342346
PUMP_STORAGE_PUMPING        -0.428692
Year                        -0.532803
EMBEDDED_SOLAR_CAPACITY     -0.536567
EMBEDDED_WIND_CAPACITY      -0.538550
Name: ND, dtype: float64

In [18]:
# Cell 6: Crear modelo final
final_model = finalize_model(best_model[0])

In [19]:
# Cell 7: Guardar modelo
save_model(final_model, os.path.join(MODEL_PATH, 'ND_model'))

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('target_transformation',
                  TransformerWrapperWithInverse(transformer=TargetTransformer(estimator=PowerTransformer(standardize=False)))),
                 ('numerical_imputer',
                  TransformerWrapper(include=['TSD', 'EMBEDDED_WIND_GENERATION',
                                              'EMBEDDED_WIND_CAPACITY',
                                              'EMBEDDED_SOLAR_GENERATION',
                                              'EMBEDDED_SOLAR_CAPACITY',
                                              'NON_BM_ST...
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['SETTLEMENT_PERIOD'],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('rest_encoding',
                  TransformerWrapper(include=['SETTLEMENT_PERIOD'],
                 

In [ ]:
from pycaret.regression import load_model, predict_model
import pandas as pd

# Cargar modelo
MODEL_PATH = './../../data/trained_models/electricity/ND_model'
model = load_model(MODEL_PATH)

# Crear DataFrame con datos de ejemplo para 2026
future_date = pd.to_datetime("2026-01-01")
future_df = pd.DataFrame({
    'SETTLEMENT_DATE': [future_date],
    'SETTLEMENT_PERIOD': [1],
    'TSD': [50],
    'EMBEDDED_WIND_GENERATION': [2000],
    'EMBEDDED_WIND_CAPACITY': [3000],
    'EMBEDDED_SOLAR_GENERATION': [1500],
    'EMBEDDED_SOLAR_CAPACITY': [2500],
    'NON_BM_STOR': [100],
    'PUMP_STORAGE_PUMPING': [500],
    'SCOTTISH_TRANSFER': [200],
    'IFA_FLOW': [300]
})

# Generar columnas derivadas
future_df['Year'] = future_df['SETTLEMENT_DATE'].dt.year
future_df['Month'] = future_df['SETTLEMENT_DATE'].dt.month
future_df['Day'] = future_df['SETTLEMENT_DATE'].dt.day
future_df['Weekday'] = future_df['SETTLEMENT_DATE'].dt.weekday

# Eliminar SETTLEMENT_DATE
future_df_model = future_df.drop(columns=['SETTLEMENT_DATE'])

# Predicción
prediction = predict_model(model, future_df_model)

# Acceder a la predicción
print("ND prediction for 1/1/2026, period 1:")
print(prediction['prediction_label'])


Transformation Pipeline and Model Successfully Loaded


ND prediction for 1/1/2026, period 1:
0    26117.023181
Name: prediction_label, dtype: float64
